In [0]:
pip install jinja2

In [0]:
jinja_config = [
    {
        "table" : "uber.bronze.stg_rides bulk_stream_rides",
        "select" : 'bulk_stream_rides.ride_id, bulk_stream_rides.confirmation_number, bulk_stream_rides.passenger_id, bulk_stream_rides.driver_id, bulk_stream_rides.vehicle_id, bulk_stream_rides.pickup_location_id, bulk_stream_rides.dropoff_location_id, bulk_stream_rides.vehicle_type_id, bulk_stream_rides.vehicle_make_id, bulk_stream_rides.payment_method_id, bulk_stream_rides.ride_status_id, bulk_stream_rides.pickup_city_id, bulk_stream_rides.dropoff_city_id, bulk_stream_rides.cancellation_reason_id, bulk_stream_rides.passenger_name, bulk_stream_rides.passenger_email, bulk_stream_rides.passenger_phone, bulk_stream_rides.driver_name, bulk_stream_rides.driver_rating, bulk_stream_rides.driver_phone, bulk_stream_rides.driver_license, bulk_stream_rides.vehicle_model, bulk_stream_rides.vehicle_color, bulk_stream_rides.license_plate, bulk_stream_rides.pickup_address, bulk_stream_rides.pickup_latitude, bulk_stream_rides.pickup_longitude, bulk_stream_rides.dropoff_address, bulk_stream_rides.dropoff_latitude, bulk_stream_rides.dropoff_longitude, bulk_stream_rides.distance_miles, bulk_stream_rides.duration_minutes, bulk_stream_rides.booking_timestamp, bulk_stream_rides.pickup_timestamp, bulk_stream_rides.dropoff_timestamp, bulk_stream_rides.base_fare, bulk_stream_rides.distance_fare, bulk_stream_rides.time_fare, bulk_stream_rides.surge_multiplier, bulk_stream_rides.subtotal, bulk_stream_rides.tip_amount, bulk_stream_rides.total_fare, bulk_stream_rides.rating',
        "where" : ""
    },
    {
        "table" : "uber.bronze.map_vehicle_makes map_vehicle_makes",
        "select" : "map_vehicle_makes.vehicle_make",
        "where" : "",
        "on" : "bulk_stream_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id"
    },
    {
        "table" : "uber.bronze.map_vehicle_types map_vehicle_types",
        "select" : "map_vehicle_types.vehicle_type,map_vehicle_types.description,map_vehicle_types.base_rate,map_vehicle_types.per_mile,map_vehicle_types.per_minute",
        "where" : "",
        "on" : "bulk_stream_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id"
    },
    {
        "table" : "uber.bronze.map_ride_statuses map_ride_statuses",
        "select" : "map_ride_statuses.ride_status",
        "where" : "",
        "on" : "bulk_stream_rides.ride_status_id = map_ride_statuses.ride_status_id"
    },
    {
        "table" : "uber.bronze.map_payment_methods map_payment_methods",
        "select" : "map_payment_methods.payment_method, map_payment_methods.is_card, map_payment_methods.requires_auth",
        "where" : "",
        "on" : "bulk_stream_rides.payment_method_id = map_payment_methods.payment_method_id"
    },
    {
        "table" : "uber.bronze.map_cities map_cities",
        "select" : "map_cities.city as pickup_city, map_cities.state, map_cities.region, map_cities.updated_at as city_updated_at",
        "where" : "",
        "on" : "bulk_stream_rides.pickup_city_id = map_cities.city_id"
    },
    {
        "table" : "uber.bronze.map_cancellation_reasons map_cancellation_reasons",
        "select" : "map_cancellation_reasons.cancellation_reason",
        "where" : "",
        "on" : "bulk_stream_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id"
    }
]


In [0]:
from jinja2 import Template
jinja_str = """

    SELECT 
        {% for config in jinja_config %}
            {{ config.select }} 
                {% if not loop.last %}
                    ,
                {% endif %}
        {% endfor %}
    FROM 
        {% for config in jinja_config %}
            {% if loop.first %}
                {{ config.table }}
            {% else %}
                LEFT JOIN {{ config.table }} ON {{ config.on }}
            {% endif %}
        {% endfor %}
   
    WHERE 1=1
        {% for config in jinja_config %}
           {% if config.where and config.where != "" %}
               AND {{ config.where }}
           {% endif %}
        {% endfor %}
"""


template = Template(jinja_str)
rendered_template = template.render(jinja_config=jinja_config)
print(rendered_template)